# NARCISSUS recreation — Colab driver

Run this notebook on Colab (browser **or** the VS Code Google Colab extension) connected to a GPU runtime. Cells progress from sanity-checks (cheap) to attack validation (medium) to the full grid (expensive — added in a later cell once the cheaper steps pass).

**Requires:** Colab Pro, Google account with Drive access.

## 1. Mount Drive

Drive holds the dataset cache, results CSV, and any trigger checkpoints — so a session crash doesn't lose work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/narcissus-recreation'
!mkdir -p {WORK}/data {WORK}/results {WORK}/checkpoints {WORK}/triggers
!ls {WORK}

## 2. Clone (or pull) the repo

In [ ]:
%cd /content
import os
if not os.path.isdir('narcissus-recreation-proposal'):
    !git clone https://github.com/alexzkhan07/narcissus-recreation-proposal.git
else:
    %cd narcissus-recreation-proposal
    !git pull
    %cd /content
%cd narcissus-recreation-proposal
!git log --oneline -3

## 3. GPU + deps check

In [ ]:
!nvidia-smi | head -20
import torch, torchvision, pandas, matplotlib
print('torch       :', torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('torchvision :', torchvision.__version__)
print('pandas      :', pandas.__version__)
print('matplotlib  :', matplotlib.__version__)

## 4. Smoke test — 5 epochs, no-op trigger

Trains ResNet-18 on clean CIFAR-10 for 5 epochs, then evaluates `tar_acc` (clean accuracy on target class) and `asr` (with no trigger applied — should be near random ~10%).

If Tar-ACC is reasonable and ASR is in the 5–15% range, the pipeline is sound.

In [ ]:
!python3 code/train_eval.py --smoke --root {WORK}/data

## 5. Attack validation — BadNets and Blend at 50% target-class poison ratio

Trains two ResNet-18s for 30 epochs each, with the BadNets and Blend triggers planted in 50% of the target-class training images. Both attacks should land **high ASR (>>50%)** and **modest Tar-ACC drop** vs. the clean baseline; otherwise the data pipeline is wrong (most often: trigger applied in the wrong color space, or applied after normalization).

30 epochs gets us to ~88-90% clean accuracy — enough to read the curve direction without spending the full 100-epoch budget. Total time on A100: ~10-15 min for both runs.

In [ ]:
import sys
sys.path.insert(0, 'code')

from train_eval import train_and_eval, TrainConfig
from attacks.badnets import make_trigger_fn as make_badnets
from attacks.blend import make_trigger_fn as make_blend

cfg = TrainConfig(epochs=30, batch_size=128, lr=0.1)
TARGET = 2
RATIO = 0.5
SEED = 0
ROOT = f'{WORK}/data'

results = []
for name, trig in [
    ('BadNets', make_badnets(patch_size=3, location='bottom-right', color=1.0)),
    ('Blend',   make_blend(alpha=0.2, pattern_seed=1234)),
]:
    print(f'\n=== {name} (target={TARGET}, ratio={RATIO}, seed={SEED}, epochs={cfg.epochs}) ===')
    tar, asr = train_and_eval(
        trigger_fn=trig, target_class=TARGET, target_class_poison_ratio=RATIO,
        seed=SEED, root=ROOT, cfg=cfg,
    )
    print(f'  -> tar_acc={tar*100:.2f}%  asr={asr*100:.2f}%')
    results.append((name, tar, asr))

print('\nsummary:')
for name, tar, asr in results:
    print(f'  {name:8s}  tar_acc={tar*100:5.2f}%  asr={asr*100:5.2f}%')

## 6. Render Figure 3 from current CSV

Currently uses stub data. Will be real once `run_grid.py` populates the CSV in a later session.

In [ ]:
!python3 code/plot_figure3.py --csv results/figure3_results.csv --out figures/figure3.png
from IPython.display import Image
Image('figures/figure3.png')

## Next steps

After cell 5 confirms BadNets and Blend produce high ASR:

1. Wrap upstream NARCISSUS into `code/attacks/narcissus.py`.
2. Implement `code/attacks/label_consistent.py`.
3. Build `code/run_grid.py` — resumable across Colab session disconnects, appends rows to `{WORK}/results/figure3_results.csv` on Drive.
4. Re-render Figure 3 with real data.